# Exceptions and tracebacks

By the end of this lesson you can:

- read a traceback and find the line that actually matters;
- catch the one error you expect, and nothing else;
- write your own error type and turn it into a clear message.

Every example uses a small library catalogue: books, copies, loans. Later, the same patterns go into **Harbour**, the copilot we build over 21 weeks.

## Two ways a program fails

**It stops with an error.** You see the problem straight away.

**It keeps running and gives a wrong answer that looks reasonable** — a fine of `0.0`, a loan for a book that does not exist. This one is worse, because nobody notices until a user does.

Python's error system exists to make failures loud. Something **raises** an error, your code can **catch** it, and the report you get is a **traceback**.

## Three words for the whole lesson

**raise** — something went wrong, so the program creates an error.

**catch** — your code decides what to do about that error.

**traceback** — Python's report of the path the error took.

That is the whole system. Most of this lesson is about reading that report, and choosing what to catch.

In [17]:
# A tiny catalogue. "copies" is how many are on the shelf right now.
CATALOGUE = {
    "BK-001": {"title": "The Salt Road", "copies": 3},
    "BK-002": {"title": "Quiet Machines", "copies": 0},
}

def fine_per_day(total_cents: int, days: int) -> float:
    return total_cents / days

print(fine_per_day(450, 3))
# Nothing catches this one, so the cell stops here and prints a traceback.
fine_per_day(450, 0)

150.0


ZeroDivisionError: division by zero

## The three parts of a traceback

Look at what the cell just printed. It has three parts, always in this order:

**The header** — `Traceback (most recent call last):`. It tells you how to read the order.

**The frames** — one block per call, oldest first. Each shows the file, the line number, the function and the line of code.

**The last line** — `ZeroDivisionError: division by zero`. The error type, then its message.

## Read the last line first

The last line is the only part that says **what** broke. Here: `ZeroDivisionError`, and the message tells you it was a division.

The frames above it say **how the program got there** — which functions called which. Useful, but not the first thing to read.

So the habit is: **last line first, then walk up the frames** until you reach code you wrote.

In [ ]:
import traceback


def find_copies(book_id: str, catalogue: dict) -> int:
    # The KeyError starts on this line when the id is not in the catalogue.
    return int(catalogue[book_id]["copies"])

def can_lend(book_id: str, catalogue: dict) -> bool:
    return find_copies(book_id, catalogue) > 0

def borrow(book_id: str, catalogue: dict) -> str:
    if can_lend(book_id, catalogue):
        return f"Loan created for {book_id}"
    return f"Sorry, {book_id} is out on loan"

# Caught only so the notebook keeps running. The traceback still prints in full.
try:
    borrow("BK-999", CATALOGUE)
except Exception as e:
    print(f"Book ID {e} not found in catalogue.")
    # traceback.print_exc()

Book ID 'BK-999' not found in catalogue.


## Walking up that traceback

| Frame | Where | What it tells you |
|---|---|---|
| Last | `find_copies` | Where Python gave up. `"BK-999"` is not a key. |
| 3rd | `can_lend` | Passes the id along. Nothing wrong here. |
| 2nd | `borrow` | The decision: *can we lend this?* |
| 1st | the call | Where `"BK-999"` came from. This is the bad value. |

`KeyError` means "this dictionary has no such key". True — but it is not the answer the caller needs. They asked whether we have the book.

In [31]:
def load_loan_file(path: str) -> list[str]:
    # The error is raised here, by open(), before we ever read a line.
    with open(path) as handle:
        return handle.read().splitlines()


def overdue_count(path: str) -> int:
    return len(load_loan_file(path))


# A different error, and the same reading method works on it.
try:
    overdue_count("loans-2026.csv")
except FileNotFoundError as e:
    print(f"File {e.filename} not found.")
    # traceback.print_exc()

File loans-2026.csv not found.


## The line that broke is not always the line that is wrong

In the first traceback, the crash was inside `fine_per_day`. But the real problem was a loan with `days = 0`. The division was innocent; the input was not.

**The drill, every time:**

1. Read the last line — what broke.
2. Walk up to the deepest frame in your own code.
3. Look at the values there. The bad value is usually one frame higher.
4. Only then open the file.

## Catching an error

`try` means: *run this line. If this particular error happens, do that instead.*

```python
try:
    risky()
except ValueError:
    recover()
```

The important word is **particular**. You name the error you expect. Anything else still reaches you, which is what you want for the bugs you have not written yet.

In [30]:
def fine_per_day(total_cents: int, days: int) -> float:
    return total_cents / days


def safe_fine_per_day(total_cents: int, days: int) -> float:
    try:
        return fine_per_day(total_cents, days)
    except ZeroDivisionError:
        # print(f"Error: {e}")
        # A zero-day loan is a real case, not a bug: there is simply no fine.
        print("No days to divide by, so there is no fine to report.")
        return 0.0


print(safe_fine_per_day(450, 3))
print(safe_fine_per_day(450, 0))
print("The program carried on.")

150.0
Error: division by zero
No days to divide by, so there is no fine to report.
0.0
The program carried on.


## Name the exact error

```python
except Exception:      # everything
except ZeroDivisionError:   # one known case
```

The first version catches real bugs, typos and wrong types as well. They all turn into `0.0`, and you never hear about them again.

The second version says: *I know about this one case and I know what to do here.* Everything else is still allowed to fail loudly.

In [27]:
def show(label: str, action) -> None:
    try:
        action()
    except Exception as exc:
        print(f"{label:<16} {type(exc).__name__:<18} {exc}")


show("wrong type", lambda: 450 / "2")
show("wrong value", lambda: int("BK-001"))
show("missing key", lambda: CATALOGUE["BK-999"])
show("missing file", lambda: open("loans-2026.csv"))

wrong type       TypeError          unsupported operand type(s) for /: 'int' and 'str'
wrong value      ValueError         invalid literal for int() with base 10: 'BK-001'
missing key      KeyError           'BK-999'
missing file     FileNotFoundError  [Errno 2] No such file or directory: 'loans-2026.csv'


## Never write bare `except:`

Bare `except:` catches `BaseException`, which is the parent of everything — including `KeyboardInterrupt` (Ctrl-C) and `SystemExit`.

So your program stops responding to Ctrl-C, and it also swallows your own mistakes. The next cell shows what that costs.

In [36]:
def fine_swallowing(total_cents: int, days: int) -> float:
    try:
        return total_cents / dayz        # typo on purpose: NameError
    except:                               # catches the typo too
        return 0.0

def fine_honest(total_cents: int, days: int) -> float:
    try:
        return total_cents / days
    except ZeroDivisionError:
        return 0.0

print(f"with bare except: {fine_swallowing(450, 3)}")
print(f"with bare except: {fine_swallowing(450, 0)}")
print(f"with a named error: {fine_honest(450, 3)}")
print(f"with a named error: {fine_honest(450, 0)}")
print()
print("The typo returned 0.0, which looks like a real fine. Nobody chases a zero.")

with bare except: 0.0
with bare except: 0.0
with a named error: 150.0
with a named error: 0.0

The typo returned 0.0, which looks like a real fine. Nobody chases a zero.


## `else` and `finally`

| Block | Runs when |
|---|---|
| `try` | always, first |
| `except` | only if that error happened |
| `else` | only if `try` finished with no error |
| `finally` | always, whatever happened |

`else` keeps the protected part small: those lines are *not* the risky part.

`finally` is for cleanup that must happen once: close the file, release the lock, roll back.

In [43]:
def order_of_blocks(days: int) -> str:
    try:
        print("  1. try")
        per_day = 450 / days
    except ZeroDivisionError:
        print("  2. except")
        return "handled"
    except TypeError:
        print("  2. except TypeError")
        return "handled"
    else:
        print("  2. else - no error, so this runs")
        return f"fine is {per_day:.1f} cents/day"
    finally:
        print("  3. finally - always")

print("days = 3 ->", order_of_blocks(3))
print()
print("days = 0 ->", order_of_blocks('0'))

  1. try
  2. else - no error, so this runs
  3. finally - always
days = 3 -> fine is 150.0 cents/day

  1. try
  2. except TypeError
  3. finally - always
days = 0 -> handled


In [45]:
def returns_from_try() -> str:
    try:
        return "returned from try"
    finally:
        print("  finally still ran after the return")

print(returns_from_try())

def handler_fails() -> None:
    try:
        raise ValueError("first error")
    except ValueError:
        # Even when the handler itself breaks, cleanup still happens.
        raise RuntimeError("handler failed too")
    finally:
        print("  finally still ran")

try:
    handler_fails()
except RuntimeError as exc:
    print(f"  the caller sees: {type(exc).__name__}: {exc}")

  finally still ran after the return
returned from try
  finally still ran
  the caller sees: RuntimeError: handler failed too


## Four rules for handlers

1. **Catch the narrowest type you can act on.**
2. **Never swallow silently.** `except Exception: pass` turns a bug into a wrong answer.
3. **Do not use errors for normal flow in a hot loop.** If it happens most of the time, use `if`.
4. **Never write bare `except:`.**

If you cannot say what you would do differently after catching it, you are not handling the error — you are hiding it.

## Two ways to guard a step: LBYL and EAFP

**LBYL — look before you leap.** Check first, then act.

```python
if book_id in catalogue:
    lend(book_id)
```

**EAFP — easier to ask forgiveness than permission.** Try it, and handle the failure.

```python
try:
    lend(book_id)
except KeyError:
    refuse()
```

Both are normal Python. They differ in one situation, and we will look at it in a moment.

In [46]:
def title_lbyl(book_id: str, catalogue: dict) -> str:
    # Look first.
    if book_id in catalogue:
        return catalogue[book_id]["title"]
    return "unknown book"


def title_eafp(book_id: str, catalogue: dict) -> str:
    # Ask forgiveness.
    try:
        return catalogue[book_id]["title"]
    except KeyError:
        return "unknown book"


for book_id in ("BK-001", "BK-999"):
    print(f"{book_id}  LBYL: {title_lbyl(book_id, CATALOGUE):<14} EAFP: {title_eafp(book_id, CATALOGUE)}")

print()
print("Same output. For one reader at one moment, they are equivalent.")

BK-001  LBYL: The Salt Road  EAFP: The Salt Road
BK-999  LBYL: unknown book   EAFP: unknown book

Same output. For one reader at one moment, they are equivalent.


## The trade-off

| | LBYL | EAFP |
|---|---|---|
| Reads like | a condition | the real action |
| When nothing fails | an extra check | nothing extra |
| When it fails | you write the error path | Python builds an error |
| Two users at once | **gap** between check and act | one step |

The gap in the last row is the important one, and the next cell shows it.

In [48]:
# One copy on the shelf. Two clerks serve two members.
shelf = {"BK-001": ["copy-1"]}


def lend_lbyl(book_id: str, shelf: dict, copies_seen: int) -> str:
    # Acts on the count the clerk checked a moment ago.
    if copies_seen > 0:
        shelf[book_id].pop()
        return "lent"
    return "refused"


def lend_eafp(book_id: str, shelf: dict) -> str:
    # Asks the shelf itself, at the moment of lending.
    try:
        shelf[book_id].pop()
        return "lent"
    except (KeyError, IndexError):
        return "refused"


copies_seen = len(shelf["BK-001"])       # both clerks look at the shelf

print("clerk A, looked first:", lend_lbyl("BK-001", shelf, copies_seen))
try:
    print("clerk B, looked first:", lend_lbyl("BK-001", shelf, copies_seen))
except IndexError:
    print("clerk B, looked first: CRASH - the copy was already gone")
print("clerk C, asks forgiveness:", lend_eafp("BK-001", shelf))
print()
print("The check was true when it was made. It was not true when it was used.")

clerk A, looked first: lent
clerk B, looked first: CRASH - the copy was already gone
clerk C, asks forgiveness: refused

The check was true when it was made. It was not true when it was used.


## Which one to use

**LBYL when**
- the message is the product — validating input where you can say exactly what to fix;
- absence is normal, and the check is cheap.

**EAFP when**
- the check and the action must be one step, as above;
- the failure really is unusual, and the handler can be narrow.

If you pick one, say what you gave up. That sentence is the answer, not the code.

## Why you need your own error

`KeyError: 'BK-999'` says *a key was missing*. It does not say *the member asked for a book we do not stock*.

That second sentence is a rule of **your** program. The way to say it is your own error type:

```
Exception
└── LibraryError              every rule this library breaks on purpose
    ├── UnknownBookId         the id is not in the catalogue
    └── BookNotAvailable      the book exists, no copies left
```

In [4]:
class LibraryError(Exception):
    """Base class for every rule this library breaks on purpose."""


class UnknownBookId(LibraryError):
    def __init__(self, book_id: str) -> None:
        self.book_id = book_id
        super().__init__(f"No book with id {book_id!r} in the catalogue")


class BookNotAvailable(LibraryError):
    def __init__(self, book_id: str) -> None:
        self.book_id = book_id
        super().__init__(f"Book {book_id!r} has no copies available")


# One handler can catch both, because both inherit from LibraryError.
for exc in (UnknownBookId("BK-999"), BookNotAvailable("BK-002")):
    try:
        raise exc
    except LibraryError as caught:
        print(f"{type(caught).__name__:<18} {caught}")

UnknownBookId      No book with id 'BK-999' in the catalogue
BookNotAvailable   Book 'BK-002' has no copies available


## Three rules for designing one

1. **Name the condition, not the layer.** `BookNotAvailable` reads well in a test. `RepositoryException` does not.
2. **Store the value that caused it** as an attribute — `self.book_id`. Then a handler never has to read English.
3. **Subclass `Exception`, never `BaseException`** — so Ctrl-C still works.

The base class pays for itself from the second error type onwards: one catch point for the boundary, precise catches for callers that can act.

In [5]:
# Good: the id is an attribute, so code can use it directly.
try:
    raise UnknownBookId("BK-999")
except LibraryError as exc:
    print("attribute:", exc.book_id)
    print("message:  ", exc)


# Weak: the id exists only inside the message.
class UnknownBookIdMessageOnly(Exception):
    pass


try:
    raise UnknownBookIdMessageOnly("No book with id 'BK-999' in the catalogue")
except UnknownBookIdMessageOnly as exc:
    print("attribute:", getattr(exc, "book_id", "none - so we parse the string"))
    print("surgery:  ", str(exc).split("'")[1])

attribute: BK-999
message:   No book with id 'BK-999' in the catalogue
attribute: none - so we parse the string
surgery:   BK-999


## Saying what caused it

When you catch one error and raise another, tell the reader which one came first:

```python
raise UnknownBookId(book_id) from exc
```

The traceback then shows the original `KeyError`, then *"the above exception was the direct cause of the following exception"*.

Without `from exc`, the link is still printed, but worded as an accident: *"during handling of the above exception"*. Say it on purpose.

In [6]:
def shelf_copy() -> dict:
    """A fresh copy of the catalogue, so this cell can run twice."""
    return {book_id: dict(row) for book_id, row in CATALOGUE.items()}


def borrow_book(book_id: str, catalogue: dict) -> dict:
    try:
        row = catalogue[book_id]
    except KeyError as exc:
        # Translate the dictionary detail into the library's own rule.
        raise UnknownBookId(book_id) from exc

    copies = int(row["copies"])
    if copies <= 0:
        raise BookNotAvailable(book_id)

    row["copies"] = copies - 1
    return {"book_id": book_id, "title": row["title"], "copies_left": copies - 1}


try:
    borrow_book("BK-999", shelf_copy())
except UnknownBookId:
    traceback.print_exc()

NameError: name 'CATALOGUE' is not defined

## The shape that scales: worker raises, boundary catches

Two jobs, two places:

**The worker** does the work and **raises** when a rule is broken. It says what went wrong, in the language of the domain.

**The boundary** catches those errors and turns them into something the caller can act on — a status and a message, later an HTTP response.

Handling sits where you can act. Reporting sits at the edge, once.

In [7]:
def handle_borrow(book_id: str, catalogue: dict) -> tuple[int, str]:
    try:
        receipt = borrow_book(book_id, catalogue)
    except UnknownBookId as exc:
        return 404, f"Unknown book id {exc.book_id!r}."
    except BookNotAvailable as exc:
        return 409, f"{exc.book_id!r} is out on loan."
    else:
        return 200, f"Borrowed {receipt['title']!r}, {receipt['copies_left']} copies left."


shelf = shelf_copy()
for book_id in ("BK-001", "BK-002", "BK-999"):
    status, message = handle_borrow(book_id, shelf)
    print(f"  {status}  {message}")

# Only LibraryError is caught, so real bugs still reach us.
try:
    handle_borrow("BK-001", [])
except Exception as exc:
    print(f"  {type(exc).__name__}: {exc}")

NameError: name 'CATALOGUE' is not defined

In [8]:
# The same shape in Harbour: a question with no page must be refused, not guessed.
PAGES = {"refunds": "Refunds take five working days."}


class NotInDocuments(Exception):
    def __init__(self, topic: str) -> None:
        self.topic = topic
        super().__init__(f"No page about {topic!r} in the documents")


def answer_from_documents(topic: str, pages: dict) -> str:
    try:
        return pages[topic]
    except KeyError as exc:
        raise NotInDocuments(topic) from exc


def handle_question(topic: str, pages: dict) -> tuple[str, str]:
    try:
        text = answer_from_documents(topic, pages)
    except NotInDocuments as exc:
        return "no answer", f"I have nothing about {exc.topic!r}."
    else:
        return "answered", text


for topic in ("refunds", "warranty"):
    state, message = handle_question(topic, PAGES)
    print(f"  {state:>9}  {message}")

   answered  Refunds take five working days.
  no answer  I have nothing about 'warranty'.


## Cheat sheet

| Task | Do this |
|---|---|
| Read a traceback | Last line for *what*, then up the frames for *where* |
| Catch an error | `except` the narrowest type you can act on |
| Pass an error on | bare `raise` |
| Report a better one | `raise YourError(...) from exc` |
| Design an error | name the condition, store the value, subclass `Exception` |
| Report at the edge | one boundary: catch your base class, return a status |

Catch what you can act on, convert what you cannot, and never delete the evidence.